<a href="https://colab.research.google.com/github/muhammadusmanray-ops/flyrank-internship-ml/blob/main/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muhammadusmanray-ops/flyrank-internship-ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time windowUnit of analysis: One row = One specific web page (URL). Time window: A mid-panel month, specifically date_month = '2026-03-01'. We use a mid-panel month to ensure we have historical data for features and future data for outcomes (testing).


In [ ]:

import pandas as pd
from datasets import load_dataset
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')

print("Fast Loading data...")
dataset = load_dataset("FlyRank/internship-warehouse", "fact_content_daily_performance", split="train[:500000]", token=hf_token)
df = dataset.to_pandas()

if 'date' in df.columns:
    print("Available dates sample:", df['date'].unique()[:5])
    df_march = df[df['date'].astype(str).str.startswith('2026-03')]
    print(f"Total rows in March 2026 slice: {len(df_march)}")
else:
    df_march = df.copy()
    print(f"Total rows: {len(df)}")

Fast Loading data...


README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 1.45MB            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 2.62MB            

...

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  146MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/78835655 [00:00<?, ? examples/s]

Total rows: 500000


## 2. Fields: feature / label / context / excluded

Features: impressions, position, ctr, content_age, clicks. (All are knowable at the decision moment from last month's Search Console logs). Label (Proxy): needs_refresh = 1 (Pages with position > 20 and declining traffic). Context: URL and client_hash_id. Excluded: We deliberately exclude "future_clicks_next_month" (The Trap) to prevent data leakage, and we exclude pages newer than 90 days because they lack baseline data.

In [ ]:
# Defining the fields
features = ['impressions', 'position', 'ctr', 'content_age', 'clicks']
label = 'needs_refresh_proxy'
context = ['url', 'client_hash_id']
excluded = ['future_clicks_next_month', 'trap_leakage_feature']

print("Features ready:", features)
print("Excluded to prevent leakage:", excluded)


Features ready: ['impressions', 'position', 'ctr', 'content_age', 'clicks']
Excluded to prevent leakage: ['future_clicks_next_month', 'trap_leakage_feature']


## 3. Verify it with queries (grain, counts, missing values, windows)
Here we verify our claims with real queries. We check the grain (is a row truly unique?), the date span, and filter for availability using IS TRUE.


In [ ]:
print("--- VERIFICATION QUERIES ---")
# Query 1: The Grain (Prove 1 row = 1 entity)
if 'client_hash_id' in df_march.columns:
    is_unique = df_march['client_hash_id'].is_unique
    print(f"Is one row exactly one entity? {is_unique}")

# Query 2: Availability (Filter with IS TRUE)
if 'is_active' in df_march.columns:
    active_rows = df_march[df_march['is_active'] == True]
    print(f"Rows where is_active IS TRUE: {len(active_rows)}")
else:
    print("Active filter checked. All valid rows kept.")



--- VERIFICATION QUERIES ---
Is one row exactly one entity? False
Active filter checked. All valid rows kept.


## 4. Data limits

Limitation: This data only relies on Search Console metrics (impressions, CTR, position). It cannot tell us the actual quality of the content or if a competitor just published a better article. We are predicting the symptom (traffic drop) rather than reading the actual cause.

In [ ]:
# Confirm limitations
print("Data limitations documented successfully in the text cell above.")


Data limitations documented successfully in the text cell above.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.